# Entraîner votre propre assistant (LoRA) dans Google Colab

Ce notebook transforme **Llama 3.2 3B** en assistant spécialisé sur **votre** sujet, puis l'exporte au format qu'utilise Ollama sur votre instance AWS.

**Ce que vous obtenez à la fin**

- un fichier `NOM-q4_k_m.gguf` d'environ 2 Go, dans votre Google Drive, prêt pour le **Lab 6** de la formation ;
- la ligne `SYSTEM_PROMPT` à mettre dans le fichier `.env` de votre instance ;
- un fichier `connaissances.md` pour le **Lab 7** (RAG), si vous utilisez vos propres données ;
- deux mesures : la qualité des réponses avant et après l'entraînement.

**Durée** : 45 à 60 minutes, dont une grande partie d'attente. **Prérequis** : un compte Google. Aucune installation sur votre poste.

**Comment l'utiliser**

1. Exécutez les cellules **dans l'ordre**, une par une, avec le bouton ▶ à gauche de chaque cellule, ou **Maj + Entrée**.
2. Attendez qu'une cellule soit finie (le ▶ réapparaît) avant de lancer la suivante.
3. Sous chaque cellule, comparez la sortie avec le **résultat attendu** décrit juste au-dessus.
4. Si Colab vous déconnecte, reprenez à l'étape 0 : tout ce qui est en mémoire est perdu, sauf ce qui est déjà dans votre Google Drive.

| Étape | Contenu | Durée |
| --- | --- | --- |
| 0 | Activer la carte graphique | 1 min |
| 1 | Installer les bibliothèques | 1 min |
| 2 | Préparer vos données | 10 à 30 min de rédaction |
| 3 | Charger le modèle de base | 2 min |
| 4 | Mesurer le modèle avant l'entraînement | 3 min |
| 5 | Entraîner l'adaptateur LoRA | 10 à 15 min |
| 6 | Mesurer après l'entraînement | 2 min |
| 7 | Fusionner et convertir en GGUF | 10 à 15 min |
| 8 | Tester le fichier avec Ollama (facultatif) | 5 min |
| 9 | Enregistrer dans Google Drive et partager | 3 min |

## Étape 0 — Activer la carte graphique

L'entraînement demande une carte graphique (GPU). Colab en prête une gratuitement, une **NVIDIA T4**.

1. Menu **Exécution** > **Modifier le type d'exécution**.
2. Choisissez **T4 GPU**, puis **Enregistrer**.
3. Exécutez la cellule ci-dessous.

*Attendu : `Tesla T4 · 14.7 Go` ou proche. Si vous lisez « Pas de carte graphique », refaites les points 1 et 2.*

Le quota gratuit de GPU est limité, de l'ordre de quelques heures par jour. Si Colab refuse la T4, réessayez plus tard.

In [ ]:
import torch
assert torch.cuda.is_available(), "Pas de carte graphique : Exécution > Modifier le type d'exécution > T4 GPU"
p = torch.cuda.get_device_properties(0)
print(f"{p.name} · {p.total_memory / 2**30:.1f} Go")

## Étape 1 — Installer les bibliothèques

On fixe les versions utilisées pendant la préparation du lab, pour que le code se comporte comme prévu.

*Attendu : aucune ligne `ERROR`, puis `Bibliothèques prêtes`. Des avertissements jaunes sur des conflits de versions avec d'autres paquets de Colab sont sans conséquence ici.*

In [ ]:
!pip install -q "transformers==4.57.6" "peft==0.21.3" "accelerate==1.15.0" sentencepiece protobuf
import transformers, peft
print("Bibliothèques prêtes :", "transformers", transformers.__version__, "· peft", peft.__version__)

## Étape 2 — Préparer vos données

Le modèle apprend à partir d'exemples question-réponse. Chaque **fait** comprend :

- un **thème** (pour classer, et pour le fichier du RAG) ;
- une **réponse** : la phrase exacte que l'assistant devra donner ;
- **quatre formulations** de la question, pour l'entraînement ;
- **une cinquième formulation**, jamais vue à l'entraînement, pour mesurer si le modèle a vraiment appris.

Ajoutez aussi quelques **refus** : des questions sur votre sujet dont vous ne connaissez pas la réponse, avec une réponse du type « Je n'ai pas cette information ». Sans eux, le modèle invente.

**Règles pour un bon résultat**

- Visez **au moins 40 faits** ; en dessous de 10, la cellule refuse de continuer.
- Une information par fait, en une ou deux phrases.
- Variez vraiment les quatre formulations : registre familier, fautes de frappe, mots différents.
- N'écrivez que des informations exactes et publiques : l'assistant les répétera.

**Deux possibilités**

- `SOURCE = "perso"` : vos propres données, à écrire dans la cellule suivante. L'exemple fourni (un club de robotique fictif) ne compte que quelques faits : remplacez-le.
- `SOURCE = "eps"` : l'exemple de la formation, l'École Polytechnique de Sousse (58 faits), téléchargé depuis le dépôt du cours. Utile pour un premier essai.

Choisissez aussi le **nom** de votre modèle : en minuscules, sans espace ni accent.

In [ ]:
SOURCE = "perso"           # "perso" : vos données ci-dessous ; "eps" : l'exemple de Polytechnique Sousse
NOM_MODELE = "mon-assistant"

In [ ]:
# Prompt système : il sera donné au modèle à l'entraînement ET sur votre instance (fichier .env).
MON_SYSTEME = ("Tu es l'assistant du club Robotix (exemple fictif). Réponds en français, de façon concise et exacte. "
               "Si tu ne connais pas une information, dis-le et renvoie vers le bureau du club.")

# (thème, réponse, [4 questions d'entraînement], question de test)
MES_FAITS = [
    ("Le club",
     "Robotix est le club de robotique de l'école (exemple fictif). Il conçoit des robots pour des concours étudiants.",
     ["C'est quoi Robotix ?", "Présente-moi le club Robotix.", "Robotix fait quoi exactement ?", "c koi le club robotix"],
     "Peux-tu me décrire Robotix ?"),
    ("Réunions",
     "Le club se réunit chaque mercredi de 14 h à 17 h, en salle B12 (exemple fictif).",
     ["Quand se réunit le club ?", "Les réunions de Robotix ont lieu quel jour ?", "À quelle heure est la réunion du club ?", "Où se retrouve le club ?"],
     "C'est quand et où, les séances de Robotix ?"),
    ("Adhésion",
     "Pour adhérer, il suffit de venir à une réunion et de remplir la fiche d'inscription ; l'adhésion est gratuite (exemple fictif).",
     ["Comment rejoindre le club ?", "L'adhésion est-elle payante ?", "Je veux m'inscrire à Robotix, comment faire ?", "Combien coûte l'inscription au club ?"],
     "Que dois-je faire pour devenir membre ?"),
]

# Refus : (réponse, [4 questions d'entraînement], question de test). Ces faits ne vont pas dans le fichier du RAG.
MES_REFUS = [
    ("Je n'ai pas cette information. Renseignez-vous auprès du bureau du club.",
     ["Quel est le budget du club ?", "Combien d'argent a Robotix ?", "Qui finance le club ?", "Le club a-t-il des sponsors ?"],
     "Quel est le montant des subventions du club ?"),
]

Exécutez la cellule suivante : elle assemble les exemples d'entraînement et de test.

*Attendu, avec `SOURCE = "eps"` : `58 faits -> 232 exemples d'entraînement, 58 questions de test`, puis un exemple. Avec vos données, les nombres dépendent de votre liste. Un message `Attention` vous signale moins de 40 faits.*

In [ ]:
import json, random, urllib.request

if SOURCE == "eps":
    url = "https://raw.githubusercontent.com/nevermind78/llm-lab/main/finetune/donnees.py"
    espace = {}
    exec(urllib.request.urlopen(url).read().decode("utf-8"), espace)
    SYSTEME = espace["SYSTEME"]
    faits = [("", rep, qs, qt) for rep, qs, qt in espace["FAITS"][:53]]
    refus = [(rep, qs, qt) for rep, qs, qt in espace["FAITS"][53:]]
else:
    SYSTEME, faits, refus = MON_SYSTEME, MES_FAITS, MES_REFUS

for f in faits:
    assert len(f) == 4 and len(f[2]) == 4, f"Fait mal formé (il faut 4 questions d'entraînement) : {f[1][:60]}"
for r in refus:
    assert len(r) == 3 and len(r[1]) == 4, f"Refus mal formé (il faut 4 questions d'entraînement) : {r[0][:60]}"
total = len(faits) + len(refus)
assert total >= 10, f"Seulement {total} faits : il en faut au moins 10, et 40 pour un bon résultat."
assert '"' not in SYSTEME and "\n" not in SYSTEME, "Le prompt système ne doit contenir ni guillemet double ni retour à la ligne."
if total < 40:
    print(f"Attention : {total} faits seulement. Le modèle apprendra mal ; visez au moins 40.")


def exemple(question, reponse):
    return [{"role": "system", "content": SYSTEME},
            {"role": "user", "content": question},
            {"role": "assistant", "content": reponse}]


tous = [(rep, qs, qt) for _, rep, qs, qt in faits] + refus
train = [exemple(q, rep) for rep, qs, _ in tous for q in qs]
test = [exemple(qt, rep) for rep, _, qt in tous]
random.Random(0).shuffle(train)
print(f"{total} faits -> {len(train)} exemples d'entraînement, {len(test)} questions de test")
print("Exemple :", json.dumps(train[0][1:], ensure_ascii=False)[:300])

## Étape 3 — Charger le modèle de base

On télécharge **Llama 3.2 3B Instruct** (6,4 Go) depuis Hugging Face, sans compte. La cellule définit aussi le format exact du prompt : c'est celui qu'Ollama utilisera sur votre instance, ce qui garantit que le modèle verra la même chose à l'entraînement et en service.

*Attendu, après 1 à 3 minutes : `Modèle chargé en float16 · 6.0 Go sur la carte` ou proche. Sur une T4, le format est `float16` ; sur une carte plus récente, `bfloat16`.*

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODELE_BASE = "unsloth/Llama-3.2-3B-Instruct"     # copie publique de Llama 3.2 3B Instruct
# bfloat16 sur les cartes récentes ; float16 sur une T4, qui ne calcule pas nativement en bfloat16
FORMAT = torch.bfloat16 if torch.cuda.get_device_capability()[0] >= 8 else torch.float16
EOT = "<|eot_id|>"

tok = AutoTokenizer.from_pretrained(MODELE_BASE)
tok.pad_token = "<|finetune_right_pad_id|>"
modele = AutoModelForCausalLM.from_pretrained(MODELE_BASE, dtype=FORMAT, device_map={"": 0})


def entete(role):
    return f"<|start_header_id|>{role}<|end_header_id|>\n\n"


def prompt_ollama(messages):
    # Texte produit par le gabarit Ollama de llama3.2, jusqu'au début de la réponse
    systeme = next((m["content"] for m in messages if m["role"] == "system"), "")
    texte = entete("system") + "Cutting Knowledge Date: December 2023\n\n" + systeme + EOT
    for m in messages:
        if m["role"] == "user":
            texte += entete("user") + m["content"] + EOT
    return texte + entete("assistant")


FIN = tok.convert_tokens_to_ids(EOT)


def repondre(question, max_tokens=200):
    messages = [{"role": "system", "content": SYSTEME}, {"role": "user", "content": question}]
    ids = torch.tensor([[tok.bos_token_id] + tok(prompt_ollama(messages), add_special_tokens=False)["input_ids"]]).cuda()
    with torch.no_grad():
        sortie = modele.generate(ids, attention_mask=torch.ones_like(ids), max_new_tokens=max_tokens,
                                 do_sample=False, eos_token_id=FIN, pad_token_id=FIN)
    return tok.decode(sortie[0, ids.shape[1]:], skip_special_tokens=True).strip()


print(f"Modèle chargé en {str(FORMAT).replace('torch.', '')} · {torch.cuda.memory_allocated() / 2**30:.1f} Go sur la carte")

## Étape 4 — Mesurer le modèle avant l'entraînement

On pose au modèle les questions de **test**, jamais vues à l'entraînement, et on calcule le **rappel** : la part des mots importants de la réponse attendue (mots de 4 lettres ou plus, nombres) que l'on retrouve dans sa réponse.

*Attendu, après 2 à 4 minutes : un rappel faible, de l'ordre de 20 % ou moins. Le modèle ne connaît pas votre sujet : il répond à côté ou invente.*

In [ ]:
import re, time, unicodedata

N_TEST = min(10, len(test))      # nombre de questions de test utilisées pour les mesures


def mots(texte):
    texte = unicodedata.normalize("NFKD", texte.lower()).encode("ascii", "ignore").decode()
    texte = re.sub(r"(?<=\d)[   ](?=\d)", "", texte)
    return {m for m in re.findall(r"[a-z0-9@._-]+", texte) if len(m) >= 4 or m.isdigit()}


def mesurer(titre):
    debut, resultats = time.time(), []
    for ex in test[:N_TEST]:
        question, reference = ex[1]["content"], ex[2]["content"]
        reponse = repondre(question)
        attendus = mots(reference)
        resultats.append((question, reponse, len(attendus & mots(reponse)) / max(len(attendus), 1)))
    rappel = 100 * sum(r for *_, r in resultats) / len(resultats)
    print(f"{titre} : rappel moyen {rappel:.1f} % · réponses à plus de 80 % : "
          f"{sum(r >= 0.8 for *_, r in resultats)}/{len(resultats)} · {time.time() - debut:.0f} s")
    return resultats


avant = mesurer("Avant l'entraînement")
for question, reponse, rappel in avant[:3]:
    print(f"\nQ : {question}\nR : {reponse[:250]}  [rappel {rappel:.0%}]")

## Étape 5 — Entraîner l'adaptateur LoRA

**LoRA** n'entraîne pas les 3 milliards de paramètres du modèle : elle ajoute à côté de petites matrices, environ 24 millions de paramètres (0,75 %), et n'entraîne qu'elles. C'est ce qui rend l'entraînement possible sur une carte gratuite.

| Réglage | Valeur | Rôle |
| --- | --- | --- |
| `RANG` | 16 | Taille des matrices LoRA : plus grand, plus de capacité, plus de mémoire |
| `EPOQUES` | 6 | Nombre de passages sur tous les exemples |
| `LOT` | 4 | Exemples traités ensemble ; mettez 2 en cas d'erreur `CUDA out of memory` |
| `TAUX` | 0,0002 | Vitesse d'apprentissage |

*Attendu, après 10 à 15 minutes sur une T4 : six lignes `époque`, avec une perte qui descend de plus de 1 à moins de 0,05, et une mémoire d'environ 11 Go. Pendant la préparation, sur 58 faits, en float16 : 1,44 · 0,26 · 0,07 · 0,02 · 0,006 · 0,003.*

Si la perte ne descend pas sous 0,1, vos réponses sont peut-être trop longues ou trop peu nombreuses : relisez la règle de l'étape 2.

In [ ]:
import math, random, time
from peft import LoraConfig, get_peft_model

RANG, EPOQUES, LOT, TAUX = 16, 6, 4, 2e-4


def encoder(messages):
    # La perte ne porte que sur la réponse de l'assistant
    debut = [tok.bos_token_id] + tok(prompt_ollama(messages[:-1]), add_special_tokens=False)["input_ids"]
    reponse = tok(messages[-1]["content"] + EOT, add_special_tokens=False)["input_ids"]
    return debut + reponse, [-100] * len(debut) + reponse


exemples = [encoder(m) for m in train]
modele = get_peft_model(modele, LoraConfig(
    r=RANG, lora_alpha=2 * RANG, lora_dropout=0.05, task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]))
modele.print_trainable_parameters()

optimiseur = torch.optim.AdamW([p for p in modele.parameters() if p.requires_grad], lr=TAUX, weight_decay=0.0)
pas_total = EPOQUES * math.ceil(len(exemples) / LOT)
planificateur = torch.optim.lr_scheduler.OneCycleLR(optimiseur, max_lr=TAUX, total_steps=pas_total, pct_start=0.1)
echelle = torch.amp.GradScaler("cuda", enabled=(FORMAT == torch.float16))   # nécessaire en float16 (T4)


def lot_tenseurs(lot):
    longueur = max(len(i) for i, _ in lot)
    ids = torch.full((len(lot), longueur), tok.pad_token_id)
    labels = torch.full((len(lot), longueur), -100)
    for k, (i, l) in enumerate(lot):
        ids[k, :len(i)] = torch.tensor(i)
        labels[k, :len(l)] = torch.tensor(l)
    return ids.cuda(), (ids != tok.pad_token_id).cuda(), labels.cuda()


modele.train()
torch.cuda.reset_peak_memory_stats()
debut = time.time()
for epoque in range(EPOQUES):
    random.Random(epoque).shuffle(exemples)
    pertes = []
    for k in range(0, len(exemples), LOT):
        ids, masque, labels = lot_tenseurs(exemples[k:k + LOT])
        with torch.autocast("cuda", dtype=FORMAT):
            perte = modele(input_ids=ids, attention_mask=masque, labels=labels).loss
        echelle.scale(perte).backward()
        echelle.unscale_(optimiseur)
        torch.nn.utils.clip_grad_norm_(modele.parameters(), 1.0)
        echelle.step(optimiseur)
        echelle.update()
        planificateur.step()
        optimiseur.zero_grad()
        pertes.append(perte.item())
    print(f"époque {epoque + 1}/{EPOQUES} · perte moyenne {sum(pertes) / len(pertes):.3f} · "
          f"{time.time() - debut:.0f} s · mémoire {torch.cuda.max_memory_allocated() / 2**30:.1f} Go", flush=True)
modele.eval()

## Étape 6 — Mesurer après l'entraînement

Mêmes questions de test, même calcul.

*Attendu : un rappel nettement plus haut qu'à l'étape 4. Pendant la préparation, sur l'exemple de Polytechnique Sousse : 20 % avant, plus de 85 % après.*

In [ ]:
apres = mesurer("Après l'entraînement")
for (question, rep_avant, r_avant), (_, rep_apres, r_apres) in list(zip(avant, apres))[:3]:
    print(f"\nQ : {question}\n  AVANT ({r_avant:.0%}) : {rep_avant[:200]}\n  APRÈS ({r_apres:.0%}) : {rep_apres[:200]}")

Posez-lui vos propres questions : modifiez le texte entre guillemets et relancez la cellule autant de fois que vous voulez. Essayez une question **hors de vos données** : invente-t-il ?

In [ ]:
QUESTION = "Que peux-tu faire pour moi ?"
print(repondre(QUESTION))

## Étape 7 — Fusionner et convertir en GGUF

Ollama, sur votre instance, n'accepte pas un adaptateur LoRA séparé : il faut un modèle complet, dans le format **GGUF**, compressé en **4 bits** pour tenir dans la mémoire d'une `t3.large`.

1. **Fusion** : on intègre l'adaptateur dans le modèle (6 Go).
2. **Conversion** en GGUF 16 bits, avec les outils de llama.cpp.
3. **Quantification** en 4 bits (`Q4_K_M`) : de 6 Go à 2 Go.

*Attendu pour la cellule de fusion, après 1 à 2 minutes : `Modèle fusionné enregistré dans /content/fusionne`.*

In [ ]:
import gc
# Libérer d'abord la mémoire occupée par l'entraînement (optimiseur, gradients)
del optimiseur, planificateur, echelle, exemples
modele.zero_grad(set_to_none=True)
gc.collect(); torch.cuda.empty_cache()
fusionne = modele.merge_and_unload()
fusionne.save_pretrained("/content/fusionne", safe_serialization=True, max_shard_size="2GB")
tok.save_pretrained("/content/fusionne")
del fusionne, modele
gc.collect(); torch.cuda.empty_cache()
print("Modèle fusionné enregistré dans /content/fusionne")

*Attendu pour la conversion, après 3 à 5 minutes : une ligne `Model successfully exported to /content/f16.gguf`.*

In [ ]:
!git clone -q --depth 1 --branch b12373 https://github.com/ggml-org/llama.cpp.git /content/llama.cpp
!python /content/llama.cpp/convert_hf_to_gguf.py /content/fusionne --outfile /content/f16.gguf --outtype f16 2>&1 | tail -2

La quantification utilise le programme `llama-quantize` de llama.cpp. La cellule essaie d'abord la version précompilée ; si elle ne fonctionne pas sur la machine de Colab, elle la compile, ce qui ajoute 5 à 10 minutes.

*Attendu : `Fichier prêt : /content/NOM-q4_k_m.gguf · 1.9 Go`.*

In [ ]:
import glob, os, subprocess, tarfile, urllib.request

TAG = "b12373"
SORTIE = f"/content/{NOM_MODELE}-q4_k_m.gguf"


def quantifier(programme):
    env = dict(os.environ, LD_LIBRARY_PATH=os.path.dirname(programme))
    r = subprocess.run([programme, "/content/f16.gguf", SORTIE, "Q4_K_M"], capture_output=True, text=True, env=env)
    return r.returncode == 0 and os.path.exists(SORTIE)


url = f"https://github.com/ggml-org/llama.cpp/releases/download/{TAG}/llama-{TAG}-bin-ubuntu-x64.tar.gz"
urllib.request.urlretrieve(url, "/content/llama-bin.tar.gz")
tarfile.open("/content/llama-bin.tar.gz").extractall("/content/llama-bin")
programme = next(iter(glob.glob("/content/llama-bin/**/llama-quantize", recursive=True)), None)

if not (programme and quantifier(programme)):
    print("Version précompilée inutilisable ici : compilation de llama-quantize (5 à 10 minutes)…")
    subprocess.run("cmake -S /content/llama.cpp -B /content/llama.cpp/build -DLLAMA_CURL=OFF -DGGML_NATIVE=ON > /dev/null "
                   "&& cmake --build /content/llama.cpp/build --target llama-quantize -j 4 > /dev/null", shell=True, check=True)
    programme = glob.glob("/content/llama.cpp/build/**/llama-quantize", recursive=True)[0]
    assert quantifier(programme), "La quantification a échoué"

os.remove("/content/f16.gguf")
print(f"Fichier prêt : {SORTIE} · {os.path.getsize(SORTIE) / 2**30:.1f} Go")

## Étape 8 — Tester le fichier avec Ollama (facultatif)

Cette étape installe Ollama dans Colab et lui fait lire votre fichier, exactement comme sur votre instance au Lab 6. Elle permet de vérifier le fichier avant de le télécharger sur AWS.

*Attendu : `success`, puis deux réponses proches de celles de l'étape 6. Ici, Ollama utilise la carte graphique de Colab ; sur votre instance, sans carte graphique, chaque réponse prendra 20 à 30 secondes.*

In [ ]:
!apt-get -qq install -y zstd > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh > /dev/null
import subprocess, time, json, urllib.request
serveur = subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(5)

# Même gabarit de prompt que llama3.2:3b, celui que creer_modele.sh reprend sur l'instance
GABARIT = r'''TEMPLATE """<|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023

{{ if .System }}{{ .System }}
{{- end }}
{{- if .Tools }}When you receive a tool call response, use the output to format an answer to the orginal user question.

You are a helpful assistant with tool calling capabilities.
{{- end }}<|eot_id|>
{{- range $i, $_ := .Messages }}
{{- $last := eq (len (slice $.Messages $i)) 1 }}
{{- if eq .Role "user" }}<|start_header_id|>user<|end_header_id|>
{{- if and $.Tools $last }}

Given the following functions, please respond with a JSON for a function call with its proper arguments that best answers the given prompt.

Respond in the format {"name": function name, "parameters": dictionary of argument name and its value}. Do not use variables.

{{ range $.Tools }}
{{- . }}
{{ end }}
{{ .Content }}<|eot_id|>
{{- else }}

{{ .Content }}<|eot_id|>
{{- end }}{{ if $last }}<|start_header_id|>assistant<|end_header_id|>

{{ end }}
{{- else if eq .Role "assistant" }}<|start_header_id|>assistant<|end_header_id|>
{{- if .ToolCalls }}
{{ range .ToolCalls }}
{"name": "{{ .Function.Name }}", "parameters": {{ .Function.Arguments }}}{{ end }}
{{- else }}

{{ .Content }}
{{- end }}{{ if not $last }}<|eot_id|>{{ end }}
{{- else if eq .Role "tool" }}<|start_header_id|>ipython<|end_header_id|>

{{ .Content }}<|eot_id|>{{ if $last }}<|start_header_id|>assistant<|end_header_id|>

{{ end }}
{{- end }}
{{- end }}"""
PARAMETER stop <|start_header_id|>
PARAMETER stop <|end_header_id|>
PARAMETER stop <|eot_id|>'''
with open("/content/Modelfile", "w", encoding="utf-8") as f:
    f.write(f"FROM {SORTIE}\n" + GABARIT + "\n")
!ollama create {NOM_MODELE} -f /content/Modelfile 2>&1 | tail -1

for ex in test[:2]:
    corps = {"model": NOM_MODELE, "stream": False, "messages": ex[:2]}
    req = urllib.request.Request("http://localhost:11434/api/chat", data=json.dumps(corps).encode(),
                                 headers={"content-type": "application/json"})
    reponse = json.load(urllib.request.urlopen(req, timeout=600))["message"]["content"]
    print(f"\nQ : {ex[1]['content']}\nR : {reponse}")

## Étape 9 — Enregistrer dans Google Drive et partager

1. Exécutez la cellule ci-dessous. Colab demande l'autorisation d'accéder à votre Google Drive : acceptez.
2. *Attendu : `Copié dans Google Drive : MyDrive/llm-lab/NOM-q4_k_m.gguf`. La copie de 2 Go prend 1 à 2 minutes.*

In [ ]:
import os, shutil
from google.colab import drive
drive.mount("/content/drive")
dossier = "/content/drive/MyDrive/llm-lab"
os.makedirs(dossier, exist_ok=True)
shutil.copy(SORTIE, dossier)
print(f"Copié dans Google Drive : MyDrive/llm-lab/{os.path.basename(SORTIE)}")

if SOURCE == "perso":
    lignes = ["# Connaissances de l'assistant", "",
              "> Fichier produit par le notebook d'entraînement. Un titre « ## » par thème, un paragraphe par information.", ""]
    for theme in dict.fromkeys(t for t, *_ in faits):
        lignes += [f"## {theme}", ""]
        lignes += [x for _, rep, *_ in (f for f in faits if f[0] == theme) for x in (rep, "")]
    with open(f"{dossier}/connaissances.md", "w", encoding="utf-8") as f:
        f.write("\n".join(lignes))
    print("Copié dans Google Drive : MyDrive/llm-lab/connaissances.md (pour le Lab 7)")

3. Ouvrez [Google Drive](https://drive.google.com), dossier **llm-lab**. Faites un clic droit sur le fichier `.gguf` > **Partager** > **Partager**.
4. Sous **Accès général**, choisissez **Tous les utilisateurs disposant du lien**, rôle **Lecteur**, puis **Copier le lien** et **OK**.
5. Le lien a la forme `https://drive.google.com/file/d/IDENTIFIANT/view?usp=sharing`. Notez l'**IDENTIFIANT**, la partie entre `/d/` et `/view`.

La dernière cellule affiche les commandes du Lab 6 adaptées à votre modèle. Collez votre identifiant entre les guillemets, puis exécutez-la.

In [ ]:
ID_DRIVE = "COLLEZ-ICI-L-IDENTIFIANT"

print(f'''Commandes du Lab 6 pour votre modèle, à taper sur l'instance (Session Manager, après sudo -i et cd /opt/llm) :

Tâche 2 — télécharger :
  ID={ID_DRIVE}
  curl -L -o /tmp/eps-assistant.gguf "https://drive.usercontent.google.com/download?id=$ID&export=download&confirm=t"

Tâche 3 — créer le modèle sous le nom {NOM_MODELE} :
  docker compose cp /tmp/eps-assistant.gguf ollama:/tmp/eps-assistant.gguf
  docker compose cp finetune/creer_modele.sh ollama:/tmp/creer_modele.sh
  docker compose exec ollama sh /tmp/creer_modele.sh /tmp/eps-assistant.gguf {NOM_MODELE}

Tâche 4 — basculer l'application (au lieu des deux commandes sed et grep de la fiche) :
  sed -i 's|^MODEL=.*|MODEL={NOM_MODELE}|' .env
  sed -i '/^SYSTEM_PROMPT=/d' .env
  cat >> .env <<'FIN'
SYSTEM_PROMPT="{SYSTEME}"
FIN
  docker compose up -d --no-build

Tâches 1, 4 et 6 : posez vos propres questions à la place de celles de la fiche.
Tâche 5 : sautez-la si vos données ne portent pas sur Polytechnique Sousse ; vos mesures sont celles des étapes 4 et 6
du notebook (le fichier test.jsonl de l'instance contient les questions de l'école).''')
if SOURCE == "perso":
    print("\nLab 7 : remplacez le contenu de app/connaissances.md, dans votre dépôt GitHub, par celui du fichier "
          "MyDrive/llm-lab/connaissances.md, puis faites un commit. Les questions de la tâche 4 et 5 seront les vôtres.")

## Pour finir

- **Libérez la carte graphique** : menu **Exécution** > **Déconnecter et supprimer l'environnement d'exécution**. Votre fichier reste dans Google Drive.
- **Ce que vous avez fait** : préparer des données, entraîner un adaptateur LoRA, mesurer le gain sur des questions jamais vues, fusionner, convertir et compresser le modèle pour un serveur sans carte graphique.

**Questions de synthèse**

1. Pourquoi mesurer sur une cinquième formulation, jamais vue à l'entraînement ?
2. L'adaptateur LoRA compte environ 24 millions de paramètres, le modèle 3,2 milliards. Pourquoi est-ce suffisant pour apprendre vos faits ?
3. Votre modèle invente-t-il sur une question hors de vos données ? Quel est le rôle des refus de l'étape 2 ?
4. Le fichier passe de 6 Go à 2 Go en 4 bits. Qu'a-t-on perdu, et pourquoi l'accepte-t-on sur une `t3.large` ?
5. Si une de vos informations change, que faut-il refaire ? Comparez avec le RAG du Lab 7.